In [5]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

Here I visualize module genes to try to decide on how to clean up the bulk data prior to running FM again (to hopefully boost signal of certain cell types)

In [6]:
options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)

In [24]:
mod_def <- "ModSeed"

max_qval <- 1e-10

In [36]:
enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"
top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)
top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]

## Prep data

### Prep enrichments & bulk data

In [ ]:
# *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

bulk_data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed" 
bulk_expr_file <- "GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv"
bulk_expr <- fread(bulk_expr_file, data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

In [9]:
# top_mods_df$Cell_type <- gsub("/", "_", top_mods_df$Cell_type, fixed=TRUE)
# top_mods_df$Cell_type <- gsub(" ", "_", top_mods_df$Cell_type, fixed=TRUE)

In [10]:
# zero.values <- sum(bulk_expr[,-1] <= 0)

# if(zero.values > 0) {
#     cat("\n")
#     print("Gene expression values <= 0 are present. Scaling all data so minimum expression = 1",quote=F)
#     cat("\n")
#     bulk_expr[,-1] <- bulk_expr[,-1] + abs(min(bulk_expr[,-1], na.rm=T)) + 1
# } 

### Prep single cell data

In [ ]:
# sc_data_source <- "yao_2021_MOp_STAR_gene_counts_normalized"

# sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_gene_counts.csv", data.table=FALSE)
# sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_sampleinfo.csv", data.table=FALSE)
# colnames(sc_expr)[1] <- "Gene"

# # Remove cell types with only a few cells

# min_cells <- 5

# ctype_tally <- table(sampleinfo$subclass_label)
# ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
# cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

# sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
# sampleinfo <- sampleinfo[cells_to_keep,]

# all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

# total_expr <- colSums(sc_expr[,-1])
# sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
# sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

In [42]:
sc_data_source <- "yao_2021_ACA_STAR_gene_counts_normalized"

sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_gene_counts.csv", data.table=FALSE)
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_sampleinfo.csv", data.table=FALSE)
colnames(sc_expr)[1] <- "Gene"

# Remove cell types with only a few cells

min_cells <- 5

ctype_tally <- table(sampleinfo$subclass_label)
ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
sampleinfo <- sampleinfo[cells_to_keep,]

all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

total_expr <- colSums(sc_expr[,-1])
sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

[1] TRUE

In [43]:
ctype_assignment_vec <- sampleinfo$subclass_label

### Prep markers

In [15]:
marker_source <- "Claude_marker_genes"

human_marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")
mouse_marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")

names(marker_genes_list)

ERROR: Error: object 'marker_genes_list' not found


In [ ]:
# marker_genes_list <- lapply(marker_genes_list, function(x) {
#     if (length(x) > 20) {
#         x[1:20]
#     } else x
# }

## Select modules

In [37]:
head(top_mods_df[,c("Cell_type", "Qval")], 25)

,Cell_type,Qval
,<chr>,<dbl>
1,kelley_microglia,7.973090e-226
2,kelley_astro,1.476741e-204
3,kelley_oligo,2.397437e-177
4,kelley_Mural,5.814351e-145
5,kelley_Endothelial,8.579244e-114
6,JAKEL_2019_IMMUNE,5.456243e-108
7,kelley_neuron,1.224765e-96
8,ZHANG_MOUSE_MICROGLIA_2014,8.326425e-92
9,JAKEL_2019_OLIGODENDROCYTE4,2.537779e-69


In [38]:
top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df_subset)

[1] 89  7

## Visualize genes

#### Module genes

In [44]:
filename <- paste0("figures/module_genes/", enrichment_source, "_", mod_def, "_", sc_data_source, "_projections.pdf")

max_genes <- 15

pdf(file=filename)

for (i in 1:nrow(top_mods_df_subset)) {
    print(paste("Module", i))

    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(
        top_mods_df_subset$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df_subset$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    # Plot gene expression over bulk samples
    
    mod_genes_subset <- na.omit(mod_genes[1:max_genes])
    plot_gene_expr_over_samples(bulk_expr, mod_genes_subset, plot_title, plot_sub, target_species=NULL)

    # Plot module genes in single cell data

    plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title, plot_sub, target_species="mouse")
}

dev.off()

[1] "Module 1"
[1] "Module 2"
[1] "Module 3"
[1] "Module 4"
[1] "Module 5"


Warning message:
“Removed 17 rows containing missing values or values outside the scale range
(`geom_bar()`).”


[1] "Module 6"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”


[1] "Module 7"


...

Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”


[1] "Module 73"
[1] "Module 74"
[1] "Module 75"
[1] "Module 76"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”


[1] "Module 77"
[1] "Module 78"
[1] "Module 79"
[1] "Module 80"
[1] "Module 81"
[1] "Module 82"
[1] "Module 83"
[1] "Module 84"
[1] "Module 85"
[1] "Module 86"
[1] "Module 87"
[1] "Module 88"
[1] "Module 89"


agg_record_825033507 
                   2

### Marker genes

In [ ]:
filename <- paste0("figures/marker_genes/", bulk_data_source, "_and_", sc_data_source, "_", marker_source, ".pdf")

pdf(file=filename)

for (i in seq_along(human_marker_genes_list)) {
    human_genes <- human_marker_genes_list[[i]]
    mouse_genes <- mouse_marker_genes_list[[i]]

    plot_title <- names(human_marker_genes_list)[i] 
    plot_sub <- NULL

    plot_gene_expr_over_samples(bulk_expr, human_genes, plot_title, plot_sub)
    plot_gene_projections(sc_expr_norm, gene_vec, ctype_assignment_vec, plot_title, plot_sub)
}

dev.off()

### Module genes